In [1]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("S3_to_Clickhouse") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        # "org.postgresql:postgresql:42.5.0,"
        # "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0"
           ) \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", os.getenv("MINIO_ROOT_USER")) \
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD")) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
ru.yandex.clickhouse#clickhouse-jdbc added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-58ccddc7-d648-451f-b33c-34501c00b351;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found ru.yandex.clickhouse#clickhouse-jdbc;0.3.2 in central
	found com.clickhouse#clickhouse-http-client;0.3.2 in central
	found com.clickhouse#clickhouse-client;0.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found com.google.code.gson#gson;2.8.8 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#c

In [2]:
hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")

In [3]:
df = spark.read.parquet("s3a://prod/stream/etl_pkg/change_dttm2=2025-06-28")

25/06/29 09:08:09 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [4]:
df.show()

+-------+--------------------+---------+-----------+
|pkg_sqn|      data_domain_id|   pkg_nm|change_dttm|
+-------+--------------------+---------+-----------+
|    484|http://www.it.ru/...|pkg_nm484|      20267|
|    485|http://www.it.ru/...|pkg_nm485|      20267|
|    486|http://www.it.ru/...|pkg_nm486|      20267|
|    487|http://www.it.ru/...|pkg_nm487|      20267|
|    488|http://www.it.ru/...|pkg_nm488|      20267|
|    489|http://www.it.ru/...|pkg_nm489|      20267|
|    490|http://www.it.ru/...|pkg_nm490|      20267|
|    491|http://www.it.ru/...|pkg_nm491|      20267|
|    492|http://www.it.ru/...|pkg_nm492|      20267|
|    493|http://www.it.ru/...|pkg_nm493|      20267|
|    494|http://www.it.ru/...|pkg_nm494|      20267|
|    495|http://www.it.ru/...|pkg_nm495|      20267|
|    496|http://www.it.ru/...|pkg_nm496|      20267|
|    497|http://www.it.ru/...|pkg_nm497|      20267|
|    498|http://www.it.ru/...|pkg_nm498|      20267|
|    499|http://www.it.ru/...|pkg_nm499|      

In [35]:
df.count()

141

In [33]:
clickhouse_url = "jdbc:clickhouse://clickhouse:8123/DM"

In [34]:
df.write \
    .format("jdbc") \
    .option("url", clickhouse_url) \
    .option("dbtable", "etl_pkg") \
    .option("user", "admin") \
    .option("password", "admin") \
    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver") \
    .mode("append") \
    .save()

25/06/29 10:28:21 WARN ClickHouseConnectionImpl: [JDBC Compliant Mode] Transaction is not supported. Change jdbcCompliant to false to throw SQLException instead.
25/06/29 10:28:21 WARN ClickHouseConnectionImpl: [JDBC Compliant Mode] Transaction is not supported. Change jdbcCompliant to false to throw SQLException instead.
25/06/29 10:28:21 WARN ClickHouseConnectionImpl: [JDBC Compliant Mode] Transaction is not supported. Change jdbcCompliant to false to throw SQLException instead.
25/06/29 10:28:21 WARN ClickHouseConnectionImpl: [JDBC Compliant Mode] Transaction is not supported. Change jdbcCompliant to false to throw SQLException instead.
25/06/29 10:28:22 WARN ClickHouseConnectionImpl: [JDBC Compliant Mode] Transaction is not supported. Change jdbcCompliant to false to throw SQLException instead.
25/06/29 10:28:22 WARN ClickHouseConnectionImpl: [JDBC Compliant Mode] Transaction [30d686c7-7710-4bce-8acb-9c73b0bc2c4b](2 queries & 0 savepoints) is committed.
25/06/29 10:28:22 WARN Click